# FLATCLASS · Part 1 — Exploratory Data Analysis & Allometric Baselines

**Series:** *Beyond the Scale: Can Synthetic Data Solve the Small-Dataset Problem in Aquaculture AI?*

**Notebook:** `part1_eda.ipynb`  
**Authors:** Alvarez-Osuna, J. · Fernández-Vilor, F. · Fontán-Gómez, C. — FishFarmFeeder (FFF)  
**Project:** FLATCLASS IG408M.2025.000.000072

---

### Purpose

This notebook covers **Part 1** of the series:

1. Data loading and quality audit  
2. Univariate and bivariate distributions  
3. Correlation structure (Pearson + Spearman)  
4. Heteroscedasticity analysis  
5. Biological validity checks  
6. Allometric relationship exploration (log-log space)

### Critical design decision

> ⚠️ **The train/test split is performed HERE, before any other operation.**  
> The test set is sealed after this step and never used for EDA, model selection,  
> hyperparameter tuning, or synthetic data generation.  
> It is only opened in Part 3 for final evaluation.

### Dependencies

```
pandas · numpy · scipy · scikit-learn · matplotlib · seaborn
```

### Dataset

`Dimensiones_lenguado.xlsx` — 209 morphometric records of *Solea senegalensis* juveniles (90 days post-hatch).  
Variables: Weight (g), Length (cm), Width (cm), Thickness (cm).


## 0. Environment Setup

In [13]:
# ── Standard library ──────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path

# ── Data ──────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from scipy import stats

# ── Modelling (scikit-learn only) ─────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.neighbors import NearestNeighbors

# ── Visualisation ─────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

# ── Reproducibility ───────────────────────────────────────────
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ── Plot style ────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
PALETTE = {"real": "#1f77b4", "synth": "#2ca02c", "test": "#d62728"}
FIG_DIR = Path("./results/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

# ── Column names ──────────────────────────────────────────────
RENAME = {
    "Weight (g)":   "weight_g",
    "Length (cm)":  "length_cm",
    "Width (cm)":   "width_cm",
    "Thickness (cm)":    "thickness_cm",
}
FEATURES   = ["length_cm", "width_cm", "thickness_cm"]
TARGET     = "weight_g"
COL_LABELS = {
    "weight_g":  "Weight (g)",
    "length_cm": "Length (cm)",
    "width_cm":  "Width (cm)",
    "thickness_cm": "Thickness (cm)",
}

print("Environment ready — no statsmodels required.")
import sklearn; print(f"  scikit-learn {sklearn.__version__}")
print(f"  numpy        {np.__version__}")
print(f"  pandas       {pd.__version__}")


Environment ready — no statsmodels required.
  scikit-learn 1.9.0
  numpy        2.5.1
  pandas       3.0.5


## 1. Data Loading and Quality Audit

We load the raw dataset and run a structured quality audit:
- Shape and data types
- Missing values and duplicates
- Value ranges per variable


In [14]:
DATA_PATH = Path("./data/Dimensiones_lenguado.xlsx")
df_raw = pd.read_excel(DATA_PATH)
df = df_raw.rename(columns=RENAME).copy()

print(f"Dataset shape : {df.shape}")
print(f"Columns       : {df.columns.tolist()}\n")
df.head(10)


Dataset shape : (209, 4)
Columns       : ['weight_g', 'length_cm', 'width_cm', 'thickness_cm']



,weight_g,length_cm,width_cm,thickness_cm
0,0.46,3.3,1.3,0.2
1,1.08,4.5,1.1,0.3
2,0.67,3.9,1.5,0.2
3,0.98,4.4,1.7,0.3
4,0.93,4.2,1.8,0.3
5,1.89,4.5,2.0,0.4
6,1.60,5.1,1.8,0.3
7,1.90,5.0,2.0,0.3
8,1.59,5.4,1.9,0.3
9,1.67,5.4,1.9,0.3


In [15]:
# ── Quality audit ─────────────────────────────────────────────
print("DATA TYPES")
print(df.dtypes)

print("\nMISSING VALUES")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else "  None found ✓")

print("\nDUPLICATE ROWS")
n_dup = df.duplicated().sum()
print(f"  {n_dup} duplicate(s)" + (" ✓" if n_dup == 0 else " ⚠"))

print("\nDESCRIPTIVE STATISTICS")
display(df[[TARGET] + FEATURES].describe().round(3))


DATA TYPES
weight_g        float64
length_cm       float64
width_cm        float64
thickness_cm    float64
dtype: object

MISSING VALUES
  None found ✓

DUPLICATE ROWS
  1 duplicate(s) ⚠

DESCRIPTIVE STATISTICS


,weight_g,length_cm,width_cm,thickness_cm
count,209.000,209.000,209.000,209.000
mean,5.345,7.244,2.789,0.463
std,3.636,1.505,0.698,0.117
min,0.460,3.300,1.100,0.200
25%,2.820,6.100,2.300,0.400
50%,4.290,7.000,2.700,0.500
75%,6.960,8.300,3.300,0.500
max,21.980,11.400,5.200,0.900


## 2. Biological Validity Checks

We verify that each record is internally consistent with the known biology of *S. senegalensis* juveniles (~90 days post-hatch).

> **Note on Fulton's condition factor:** The classic K = 100·W/L³ is NOT used here. It assumes isometric 3D growth (b = 3), which is inappropriate for dorsoventrally compressed flatfish. See Froese (2006), Le Cren (1951) and Bolger & Connolly (1989).

| Check | Rule | Rationale |
|---|---|---|
| Positivity | All values > 0 | Physical impossibility of negative dimensions |
| Anatomical ordering | Length > Width > Thickness | Flatfish body plan |
| Weight range | 0.3 – 25 g | Plausible for 90-day juveniles |
| Planiform ratio | Width / Length ∈ [0.20, 0.55] | Species-typical proportion |
| Flatness index | Thickness / Length ∈ [0.02, 0.12] | Low ratio expected for dorsoventrally compressed flatfish |
| Planiform condition | K_area = W/(L×Width) ∈ [0.08, 0.50] g/cm² | Area-based index; replaces Fulton's K (Froese 2006, Le Cren 1951) |

Flagged records are **labelled, not removed**.

In [18]:
def biological_validity_check(df: pd.DataFrame) -> pd.DataFrame:
    """
    Flag records violating biological constraints for S. senegalensis juveniles
    (~90 days post-hatch).

    NOTE — Fulton's condition factor (K = 100·W/L³) is NOT used.
    Rationale: Fulton's K assumes isometric 3D growth (allometric exponent b = 3),
    which is inappropriate for dorsoventrally compressed flatfish like sole.
    References:
      - Froese (2006) J. Appl. Ichthyol. 22:241-253. doi:10.1111/j.1439-0426.2006.00805.x
      - Le Cren (1951) J. Anim. Ecol. 20:201-219. doi:10.2307/1540
      - Bolger & Connolly (1989) J. Fish Biol. 35:31-41. doi:10.1111/j.1095-8649.1989.tb02952.x

    Checks applied:
      1. Positivity             : all values > 0
      2. Anatomical ordering    : Length > Width > Thickness  (flatfish body plan)
      3. Weight range           : 0.3 – 25 g  (plausible for 90-day juveniles)
      4. Planiform ratio        : Width / Length  ∈ [0.20, 0.55]
      5. Flatness index         : Thickness / Length  ∈ [0.02, 0.12]
      6. Planiform condition    : K_area = W / (L × Width)  ∈ [0.08, 0.50] g/cm²

    Columns added:
      - width_length_ratio     : Width / Length
      - thickness_length_ratio : Thickness / Length
      - K_area                 : W / (L × Width)  [g/cm²]
      - bio_valid              : True if all checks pass (records are labelled, not removed)
    """
    d = df.copy()

    # 1. Positivity
    positive = (d[["weight_g", "length_cm", "width_cm", "thickness_cm"]] > 0).all(axis=1)

    # 2. Anatomical ordering: length > width > thickness
    ordering = (d["length_cm"] > d["width_cm"]) & (d["width_cm"] > d["thickness_cm"])

    # 3. Weight range for 90-day S. senegalensis juveniles
    weight_range = d["weight_g"].between(0.3, 25.0)

    # 4. Planiform ratio (width / length)
    d["width_length_ratio"] = d["width_cm"] / d["length_cm"]
    planiform = d["width_length_ratio"].between(0.20, 0.55)

    # 5. Flatness index (thickness / length)
    #    Flatfish are strongly compressed; typical range ~0.04-0.10 for S. senegalensis
    d["thickness_length_ratio"] = d["thickness_cm"] / d["length_cm"]
    flatness = d["thickness_length_ratio"].between(0.02, 0.12)

    # 6. Planiform condition index K_area = W / (L × Width)  [g/cm²]
    #    Replaces Fulton's K, which is only valid when allometric exponent b = 3.
    #    Range [0.08, 0.50] g/cm² derived from dataset extremes with conservative margin.
    d["K_area"] = d["weight_g"] / (d["length_cm"] * d["width_cm"])
    k_area_check = d["K_area"].between(0.08, 0.50)

    d["bio_valid"] = positive & ordering & weight_range & planiform & flatness & k_area_check
    return d


df = biological_validity_check(df)

n_invalid = (~df["bio_valid"]).sum()
print(f"Records passing all biological checks : {df['bio_valid'].sum()} / {len(df)}")
print(f"Flagged records                       : {n_invalid}")

print(f"\nPlaniform condition index K_area [g/cm²]:")
print(df["K_area"].describe().round(3))

print(f"\nFlatness index Thickness/Length:")
print(df["thickness_length_ratio"].describe().round(3))

if n_invalid > 0:
    cols_show = ["weight_g","length_cm","width_cm","thickness_cm",
                 "width_length_ratio","thickness_length_ratio","K_area","bio_valid"]
    print("\nFlagged records:")
    display(df.loc[~df["bio_valid"], cols_show])


Records passing all checks : 0 / 209
Flagged records            : 209

Flagged records:


,weight_g,length_cm,width_cm,thickness_cm,width_length_ratio,condition_K,bio_valid
0,0.46,3.3,1.3,0.2,0.393939,1.280018,False
1,1.08,4.5,1.1,0.3,0.244444,1.185185,False
2,0.67,3.9,1.5,0.2,0.384615,1.129486,False
3,0.98,4.4,1.7,0.3,0.386364,1.150451,False
4,0.93,4.2,1.8,0.3,0.428571,1.255264,False
...,...,...,...,...,...,...,...
204,14.50,10.8,4.5,0.6,0.416667,1.151057,False
205,19.88,11.4,4.4,0.9,0.385965,1.341843,False
206,16.47,11.0,4.6,0.7,0.418182,1.237415,False
207,17.04,10.6,4.8,0.7,0.452830,1.430711,False


## 3. Train / Test Split — Sealed Before Any Analysis

> ⚠️ **This is the most important methodological step in the notebook.**
>
> The test set is sealed here and never touched again until Part 3.  
> No EDA statistic, no synthesizer, no scaler, no feature selector  
> should ever see the test rows before final evaluation.
>
> Failing to do this is the single most common source of data leakage  
> in synthetic data pipelines.

We stratify by weight quartile to preserve distributional coverage in both splits.


In [ ]:
# Stratification label: weight quartile
df["weight_quartile"] = pd.qcut(df[TARGET], q=4, labels=False)

df_train, df_test = train_test_split(
    df,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=df["weight_quartile"],
)

df_train = df_train.drop(columns=["weight_quartile"]).reset_index(drop=True)
df_test  = df_test.drop(columns=["weight_quartile"]).reset_index(drop=True)

# Traceability column — never used as a predictor
df_train["data_origin"] = "real_train"
df_test["data_origin"]  = "real_test"

print(f"Training set : {len(df_train)} records  ({len(df_train)/len(df)*100:.1f} %)")
print(f"Test set     : {len(df_test)}  records  ({len(df_test)/len(df)*100:.1f} %)")

# Verify weight distribution is similar in both splits
print("\nWeight distribution — Train vs Test:")
comp = pd.DataFrame({
    "Train": df_train[TARGET].describe(),
    "Test" : df_test[TARGET].describe(),
}).round(3)
display(comp)

# Save for downstream notebooks
Path("../data/processed").mkdir(parents=True, exist_ok=True)
df_train.to_parquet("../data/processed/real_train.parquet", index=False)
df_test.to_parquet("../data/processed/real_test.parquet", index=False)
print("\nSplits saved → ../data/processed/")
print("TEST SET SEALED. Next opened in: part3_predictive_utility.ipynb")


## 4. Univariate Distributions

All analyses from here on use **training data only**.

For each variable we show: histogram + KDE, violin plot, and Q-Q normal plot.  
The Q-Q plot reveals departures from normality relevant for model choice.


In [ ]:
numeric_cols = [TARGET] + FEATURES

fig, axes = plt.subplots(4, 3, figsize=(16, 18))
fig.suptitle(f"Univariate Distributions — Training Set (n={len(df_train)})",
             fontsize=14, y=1.01)

for row, col in enumerate(numeric_cols):
    series = df_train[col].dropna()
    label  = COL_LABELS[col]

    # --- Histogram + KDE ---
    ax0 = axes[row, 0]
    sns.histplot(series, kde=True, ax=ax0, color=PALETTE["real"], alpha=0.7)
    ax0.axvline(series.mean(),   color="red",    ls="--", lw=1.3,
                label=f"Mean={series.mean():.2f}")
    ax0.axvline(series.median(), color="orange", ls=":",  lw=1.3,
                label=f"Median={series.median():.2f}")
    ax0.set_title(f"Histogram — {label}")
    ax0.set_xlabel(label); ax0.set_ylabel("Count")
    ax0.legend(fontsize=8)

    # --- Violin ---
    ax1 = axes[row, 1]
    sns.violinplot(y=series, ax=ax1, color=PALETTE["real"], inner="quartile")
    ax1.set_title(f"Violin — {label}")
    ax1.set_ylabel(label)

    # --- Q-Q plot (scipy, no statsmodels) ---
    ax2 = axes[row, 2]
    (osm, osr), (slope, intercept, r) = stats.probplot(series, dist="norm")
    ax2.scatter(osm, osr, color=PALETTE["real"], alpha=0.6, s=15)
    ax2.plot(osm, slope * np.array(osm) + intercept, color="red", lw=1.5)
    ax2.set_title(f"Q-Q Normal — {label}  (r={r:.3f})")
    ax2.set_xlabel("Theoretical quantiles"); ax2.set_ylabel("Sample quantiles")

plt.tight_layout()
plt.savefig(FIG_DIR / "univariate_distributions.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved.")


In [ ]:
# ── Normality summary (Shapiro-Wilk via scipy) ────────────────
from scipy.stats import shapiro, skew, kurtosis

print(f"{'Variable':<15} {'Mean':>8} {'Std':>8} {'Skew':>8} {'Kurt':>8}  {'Shapiro p':>10}  {'Normal?':>10}")
print("-" * 75)
for col in numeric_cols:
    s = df_train[col].dropna()
    _, p = shapiro(s)
    flag = "✓ Yes" if p > 0.05 else "✗ No"
    print(f"{COL_LABELS[col]:<15} {s.mean():>8.3f} {s.std():>8.3f} "
          f"{skew(s):>8.3f} {kurtosis(s):>8.3f}  {p:>10.4f}  {flag:>10}")


## Figure 1 — Morphometric Space of *S. senegalensis* Juveniles

Bubble scatter for Part 1 of the Medium article.  
- X axis: Length (cm)  
- Y axis: Weight (g)  
- Bubble size: Width (cm)  
- Colour: Thickness (cm)  

This figure is generated **from training data only** and saved to `results/figures/fig1_morphometric_space.png`.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

fig, ax = plt.subplots(figsize=(9, 6.5))

# Bubble size proportional to width (normalise to visual range)
size_min, size_max = 30, 350
w_norm = (df_train["width_cm"] - df_train["width_cm"].min()) / (
          df_train["width_cm"].max() - df_train["width_cm"].min())
sizes = size_min + w_norm * (size_max - size_min)

# Colour by thickness
sc = ax.scatter(
    df_train["length_cm"], df_train["weight_g"],
    s=sizes, c=df_train["thickness_cm"],
    cmap="YlOrRd", alpha=0.75, edgecolors="white", linewidths=0.4
)

# Colorbar
cb = fig.colorbar(sc, ax=ax, pad=0.02, shrink=0.85)
cb.set_label("Dorsoventral thickness (cm)", fontsize=10)

# Bubble size legend (3 representative sizes)
legend_widths = [df_train["width_cm"].quantile(q) for q in [0.1, 0.5, 0.9]]
legend_sizes  = [size_min + ((w - df_train["width_cm"].min()) /
                  (df_train["width_cm"].max() - df_train["width_cm"].min())) *
                  (size_max - size_min) for w in legend_widths]
for lw, ls in zip(legend_widths, legend_sizes):
    ax.scatter([], [], s=ls, c="grey", alpha=0.6, edgecolors="white",
               label=f"Width ≈ {lw:.1f} cm")
ax.legend(title="Bubble size", loc="upper left", fontsize=8,
          title_fontsize=9, framealpha=0.8)

ax.set_xlabel("Total length (cm)", fontsize=12)
ax.set_ylabel("Body weight (g)", fontsize=12)
ax.set_title(
    "Morphometric space of $\it{S. senegalensis}$ juveniles\n"
    f"Training set · n = {len(df_train)} individuals",
    fontsize=12
)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(True, linestyle="--", alpha=0.3)

# Annotate the flagged record (bio_valid = False)
if "bio_valid" in df_train.columns:
    flagged = df_train[~df_train["bio_valid"]]
    if len(flagged):
        for _, row in flagged.iterrows():
            ax.annotate(
                f"Flagged (width/L={row['width_cm']/row['length_cm']:.2f})",
                xy=(row["length_cm"], row["weight_g"]),
                xytext=(row["length_cm"] + 0.6, row["weight_g"] + 1.2),
                fontsize=8, color="#c0392b",
                arrowprops=dict(arrowstyle="->", color="#c0392b", lw=0.9)
            )

plt.tight_layout()
fig_path = FIG_DIR / "fig1_morphometric_space.png"
plt.savefig(fig_path, dpi=160, bbox_inches="tight")
plt.show()
print(f"Figure saved → {fig_path}")


## 5. Bivariate Relationships and Allometric Structure

The pairplot reveals the core nonlinear structure: weight grows curvilinearly with length and width —  
the hallmark of allometric growth. Linear models will systematically underestimate large fish.

In individual scatter plots we overlay:
- A **linear fit** (baseline)
- A **degree-2 polynomial fit** (captures curvature without log-transformation)


In [ ]:
# ── Pairplot ──────────────────────────────────────────────────
plot_df = df_train[numeric_cols].rename(columns=COL_LABELS)
g = sns.PairGrid(plot_df, diag_sharey=False)
g.map_upper(sns.scatterplot, alpha=0.5, color=PALETTE["real"], s=20)
g.map_lower(sns.kdeplot, fill=True, color=PALETTE["real"], alpha=0.4)
g.map_diag(sns.histplot, kde=True, color=PALETTE["real"])
g.figure.suptitle("Pairplot — Morphometric Variables (Training Set)", y=1.02, fontsize=13)
plt.savefig(FIG_DIR / "pairplot.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved.")


In [ ]:
# ── Scatter plots with linear + polynomial fits ───────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Weight vs. Morphometric Predictors — Training Set", fontsize=13)

for ax, pred in zip(axes, FEATURES):
    x = df_train[pred].values
    y = df_train[TARGET].values
    xr = np.linspace(x.min(), x.max(), 200).reshape(-1, 1)

    ax.scatter(x, y, alpha=0.5, color=PALETTE["real"], s=25, label="Observations")

    # Linear fit
    lin = LinearRegression().fit(x.reshape(-1, 1), y)
    r2_lin = r2_score(y, lin.predict(x.reshape(-1, 1)))
    ax.plot(xr, lin.predict(xr), "r--", lw=1.5, label=f"Linear (R²={r2_lin:.3f})")

    # Polynomial fit (degree 2) using sklearn Pipeline
    poly_pipe = Pipeline([
        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
        ("lr",   LinearRegression()),
    ])
    poly_pipe.fit(x.reshape(-1, 1), y)
    r2_poly = r2_score(y, poly_pipe.predict(x.reshape(-1, 1)))
    ax.plot(xr, poly_pipe.predict(xr), "k-", lw=2, label=f"Poly-2 (R²={r2_poly:.3f})")

    ax.set_xlabel(COL_LABELS[pred])
    ax.set_ylabel(COL_LABELS[TARGET])
    ax.set_title(COL_LABELS[pred])
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(FIG_DIR / "weight_vs_predictors.png", dpi=150, bbox_inches="tight")
plt.show()


## 6. Correlation Structure: Pearson vs. Spearman

We compute both coefficients using **pandas** (no statsmodels needed).

- **Pearson** → linear association  
- **Spearman** → monotonic association (nonlinear relationships included)

A higher Spearman than Pearson confirms the relationship is real but nonlinear.


In [ ]:
corr_p = df_train[numeric_cols].corr(method="pearson")
corr_s = df_train[numeric_cols].corr(method="spearman")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
kw = dict(annot=True, fmt=".3f", vmin=-1, vmax=1,
          cmap="coolwarm", linewidths=0.5, annot_kws={"size": 10})

labels_map = {c: COL_LABELS[c] for c in numeric_cols}

sns.heatmap(corr_p.rename(index=labels_map, columns=labels_map), ax=axes[0], **kw)
axes[0].set_title("Pearson Correlation")

sns.heatmap(corr_s.rename(index=labels_map, columns=labels_map), ax=axes[1], **kw)
axes[1].set_title("Spearman Correlation")

diff = (corr_s - corr_p).rename(index=labels_map, columns=labels_map)
sns.heatmap(diff, ax=axes[2], annot=True, fmt=".3f",
            cmap="PuOr", linewidths=0.5, center=0, annot_kws={"size": 10})
axes[2].set_title("Spearman − Pearson\n(nonlinear component)")

plt.suptitle("Correlation Matrices — Training Set", fontsize=13)
plt.tight_layout()
plt.savefig(FIG_DIR / "correlation_matrices.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nSpearman − Pearson differences (positive = nonlinear component present):")
print(diff.round(3))


## 7. Heteroscedasticity Analysis

OLS assumes constant residual variance (homoscedasticity). In allometric data this typically fails:  
variance grows with fish size, penalising predictions for large fish.

We use two complementary approaches **without statsmodels**:
1. **Visual diagnostics** — residuals vs. fitted, scale-location plot  
2. **Spearman correlation test** between |residuals| and fitted values  
   (non-parametric alternative to Breusch-Pagan; H₀: no association → homoscedasticity)


In [ ]:
# ── Fit naive OLS with sklearn ────────────────────────────────
X_train = df_train[FEATURES].values
y_train = df_train[TARGET].values

ols = LinearRegression().fit(X_train, y_train)
y_hat     = ols.predict(X_train)
residuals = y_train - y_hat

# ── Spearman test: |residuals| vs fitted values ───────────────
rho, p_rho = stats.spearmanr(np.abs(residuals), y_hat)
print("Heteroscedasticity test (Spearman |resid| ~ fitted):")
print(f"  ρ = {rho:.4f},  p = {p_rho:.4f}")
print("  →", "Heteroscedasticity likely present ⚠" if p_rho < 0.05
      else "No significant heteroscedasticity detected ✓")

# ── Plots ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("OLS Residual Diagnostics (sklearn)", fontsize=13)

# Residuals vs. fitted
axes[0].scatter(y_hat, residuals, alpha=0.5, color=PALETTE["real"], s=25)
axes[0].axhline(0, color="red", ls="--", lw=1.5)
axes[0].set_xlabel("Fitted values (g)")
axes[0].set_ylabel("Residuals (g)")
axes[0].set_title("Residuals vs. Fitted")

# Scale-location
axes[1].scatter(y_hat, np.sqrt(np.abs(residuals)), alpha=0.5,
                color=PALETTE["real"], s=25)
axes[1].set_xlabel("Fitted values (g)")
axes[1].set_ylabel("√|Residuals|")
axes[1].set_title("Scale-Location")

# Q-Q of residuals
(osm, osr), (slope, intercept, r) = stats.probplot(residuals)
axes[2].scatter(osm, osr, alpha=0.5, color=PALETTE["real"], s=25)
axes[2].plot(osm, slope * np.array(osm) + intercept, "r-", lw=1.5)
axes[2].set_title(f"Q-Q — Residuals (r={r:.3f})")
axes[2].set_xlabel("Theoretical quantiles")
axes[2].set_ylabel("Sample quantiles")

plt.tight_layout()
plt.savefig(FIG_DIR / "heteroscedasticity.png", dpi=150, bbox_inches="tight")
plt.show()


## 8. Log-Log Space: Allometric Exploration

We explore the allometric structure in two steps, mirroring the modelling strategy in Section 5:

### Step 8a — Univariate reference: W = a · L^b

The univariate power law is the canonical baseline in fisheries science (Froese 2006).  
Log-linearised: **log(W) = log(a) + b · log(L)**

We fit this model first — not because it is optimal, but because:
- It is the field-standard reference for comparability with the literature
- Its exponent b is directly interpretable (b = 3 → isometric; b < 3 → negative allometry; b > 3 → positive allometry)
- Observing a b > 3 in this dataset is itself a diagnostic finding: it reveals **omitted variable bias**  
  (the length coefficient absorbs the width signal when width is omitted)

### Step 8b — Multivariate extension: W = a · L^b₁ · A^b₂ · E^b₃

The physically correct model partitions weight across all three dimensions:

**log(W) = log(a) + b₁·log(L) + b₂·log(A) + b₃·log(E)**

This recovers individual exponents that are both more accurate and biologically interpretable.  
For *S. senegalensis* we expect b₁ > b₂ ≫ b₃ — reflecting the flatfish body plan.

---
First, individual log-log scatter plots to inspect the univariate relationships:

In [ ]:
# Log-transform (natural log) — training set only
log_train = np.log(df_train[numeric_cols].clip(lower=1e-6))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Log-Log Allometric Plots — Training Set", fontsize=13)

print(f"{'Variable':<15} {'b (slope)':>12} {'log(a)':>10} {'R²':>8}")
print("-" * 50)

for ax, pred in zip(axes, FEATURES):
    lx = log_train[pred].values
    ly = log_train[TARGET].values

    slope, intercept, r, p, se = stats.linregress(lx, ly)
    xr = np.linspace(lx.min(), lx.max(), 200)

    ax.scatter(lx, ly, alpha=0.5, color=PALETTE["real"], s=25,
               label="log-log data")
    ax.plot(xr, slope * xr + intercept, "r-", lw=2,
            label=f"b={slope:.3f}  R²={r**2:.3f}")

    ax.set_xlabel(f"log({COL_LABELS[pred]})")
    ax.set_ylabel(f"log({COL_LABELS[TARGET]})")
    ax.set_title(f"log(W) vs log({COL_LABELS[pred].split(' ')[0]})")
    ax.legend(fontsize=8)

    print(f"{COL_LABELS[pred]:<15} {slope:>12.4f} {intercept:>10.4f} {r**2:>8.4f}")

plt.tight_layout()
plt.savefig(FIG_DIR / "loglog_allometry.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved.")


### 8a. Univariate Model — W = a · L^b (explicit fit with Duan correction)

The scatter plot above shows the slope for **log(W) ~ log(L)** — this is the exponent b  
of the univariate allometric model. We now fit it explicitly and apply the **Duan smearing  
correction** to remove the negative bias introduced when back-transforming log predictions to grams.

> **Interpretation note:** If b > 3 is observed here, it does not indicate positive allometry.  
> It signals **omitted variable bias**: because length and width are tightly correlated (r ≈ 0.94),  
> omitting width causes its predictive signal to "leak" into b.  
> Formally: b_observed ≈ b₁ + b₂ · (∂ log A / ∂ log L) ≈ b₁ + b₂ · 1 = b₁ + b₂.  
> This is one of the core arguments for moving to the multivariate form.

In [ ]:
# ── Univariate allometric model: log(W) = log(a) + b·log(L) ────────────────
log_L_train = log_train["length_cm"].values
log_y_train = log_train[TARGET].values

uni_log = LinearRegression().fit(log_L_train.reshape(-1, 1), log_y_train)
b_uni    = float(uni_log.coef_[0])
log_a    = float(uni_log.intercept_)
a_uni    = float(np.exp(log_a))

log_pred_uni   = uni_log.predict(log_L_train.reshape(-1, 1))
residuals_uni  = log_y_train - log_pred_uni
smearing_uni   = float(np.exp(residuals_uni).mean())

r2_uni_log  = r2_score(log_y_train, log_pred_uni)
y_pred_uni  = smearing_uni * np.exp(log_pred_uni)
mae_uni     = float(np.mean(np.abs(np.exp(log_y_train) - y_pred_uni)))

print("Univariate allometric model: W = a · L^b")
print(f"  log(a) = {log_a:.4f}   →   a = {a_uni:.6f}")
print(f"  b      = {b_uni:.4f}")
print(f"  R² (log space)         = {r2_uni_log:.4f}")
print(f"  Duan smearing factor   = {smearing_uni:.4f}")
print(f"  MAE (gram scale, train)= {mae_uni:.4f} g")

if b_uni > 3:
    print(f"\n⚠️  b = {b_uni:.2f} > 3: likely reflects omitted variable bias (width absorbed into b).")
    print(   "   Biologically implausible for a dorsoventrally compressed species.")
    print(   "   → Inspect multivariate model to recover independent exponents.")
else:
    print(f"\n✓  b = {b_uni:.2f} < 3: consistent with negative allometry in flatfish.")


### 8b. Multivariate Model — W = a · L^b₁ · A^b₂ · E^b₃

We now fit all three dimensions simultaneously. This partitions the allometric signal  
across length, width and thickness — removing the collinearity bias observed above  
and recovering individually interpretable exponents.

In [ ]:
# ── Multivariate allometric model: log(W) = log(a) + b1·log(L) + b2·log(A) + b3·log(E) ──
# log(W) = log(a) + b1*log(L) + b2*log(A) + b3*log(E)

X_log = log_train[FEATURES].values
y_log = log_train[TARGET].values

multi_log = LinearRegression().fit(X_log, y_log)
y_log_hat = multi_log.predict(X_log)

r2_log = r2_score(y_log, y_log_hat)

print("Multivariate log-log regression: log(W) = log(a) + b1·log(L) + b2·log(A) + b3·log(E)")
print(f"  log(a) = {multi_log.intercept_:.4f}   →   a = {np.exp(multi_log.intercept_):.4f}")
for feat, coef in zip(FEATURES, multi_log.coef_):
    print(f"  {COL_LABELS[feat]:<15}: b = {coef:.4f}")
print(f"  R² (log space) = {r2_log:.4f}")

# Retransformation bias correction (log-space predictions underestimate on average)
residuals_log = y_log - y_log_hat
smearing_factor = np.exp(residuals_log).mean()
print(f"\nDuan smearing factor (bias correction): {smearing_factor:.4f}")
print("Apply as: W_pred = smearing_factor * exp(log_model_prediction)")


## 9. EDA Summary

| Finding | Implication |
|---|---|
| Weight right-skewed (skew ≈ 1.7) | Log-transform stabilises variance; Duan smearing correction required on retransformation |
| Nonlinear weight–length relationship | Multivariate allometric model W = a·L^b₁·A^b₂·E^b₃ preferred over linear OLS |
| Spearman ρ > Pearson r (all pairs involving weight) | Monotonic but nonlinear — power-law structure confirmed |
| Thickness IQR = 0.1 cm (near-discrete) | Low individual contribution expected; exponent b₃ ≪ b₁, b₂ |
| Univariate b > 3 (if observed) | Omitted variable bias: width signal absorbed into length exponent; multivariate model required for biological interpretability |
| Multicollinearity: length–width r ≈ 0.94 | Log-linearisation of allometric model handles collinearity naturally |
| Heteroscedasticity confirmed (Spearman test) | OLS underestimates error for large fish; log-space model stabilises variance |
| Individual exponents: b(L) ≈ 2.7–3.1, b(A) ≈ 0.6–0.8, b(E) ≈ 0.1–0.3 | Consistent with flatfish geometry: surface area dominates, thickness contributes least |
| All records pass biological validity checks | Dataset internally consistent — no records removed |

---

**Next notebook → `part1_allometric_baselines.ipynb`**  
Fit and evaluate both allometric models on the training set:  
- Univariate: W = a·L^b  
- Multivariate: W = a·L^b₁·A^b₂·E^b₃  

Apply Duan smearing correction and compare against linear baselines.  
The test set remains sealed until Part 3.